In [4]:
# Zelle 1: Imports
import json

# Zelle 2: Initialisierung der Ergebnislisten
auftraege_counts = []
bestellpositionen_counts = []

# Zelle 3: Alle Monate durchgehen und zählen
for month in range(7, 11):
    for part in range(1, 3):
        filename = f"Construction_RealLife_2024_{month}_{part}_2.json"
        try:
            with open(filename, "r", encoding="utf-8") as f:
                data = json.load(f)

            auftraege = data.get("Auftraege", [])
            bestellpositionen = data.get("Bestellpositionen", [])
            workers = data.get("Arbeiter", [])
            machines = data.get("Maschinen", [])
            attachments = data.get("Anbaugeraete", [])

            num_auftraege = len(auftraege)
            num_bestellpositionen = len(bestellpositionen)
            num_workers = len(workers)
            num_machines = len(machines)
            num_attachments = len(attachments)


            auftraege_counts.append(num_auftraege)
            bestellpositionen_counts.append(num_bestellpositionen)

            print(f"Monat {month}_{part}_2:")
            print(f"  Anzahl der Aufträge: {num_auftraege}")
            print(f"  Anzahl der Bestellpositionen: {num_bestellpositionen}")
            print(f"  Anzahl der Maschinen: {num_machines}")
            print(f"  Anzahl der Arbeiter: {num_workers}")
            print(f"  Anzahl der Anbaugeräte: {num_attachments}")
            print("-" * 40)

        except FileNotFoundError:
            print(f"Datei {filename} nicht gefunden.")
        except json.JSONDecodeError:
            print(f"Datei {filename} konnte nicht als JSON geladen werden.")

# Zelle 4: Min/Max-Werte ausgeben
if auftraege_counts and bestellpositionen_counts:
    print("Gesamtübersicht:")
    print(f"  Aufträge: min = {min(auftraege_counts)}, max = {max(auftraege_counts)}")
    print(f"  Bestellpositionen: min = {min(bestellpositionen_counts)}, max = {max(bestellpositionen_counts)}")

Monat 7_1_2:
  Anzahl der Aufträge: 88
  Anzahl der Bestellpositionen: 926
  Anzahl der Maschinen: 65
  Anzahl der Arbeiter: 118
  Anzahl der Anbaugeräte: 391
----------------------------------------
Monat 7_2_2:
  Anzahl der Aufträge: 158
  Anzahl der Bestellpositionen: 2112
  Anzahl der Maschinen: 65
  Anzahl der Arbeiter: 118
  Anzahl der Anbaugeräte: 391
----------------------------------------
Monat 8_1_2:
  Anzahl der Aufträge: 149
  Anzahl der Bestellpositionen: 1717
  Anzahl der Maschinen: 65
  Anzahl der Arbeiter: 118
  Anzahl der Anbaugeräte: 368
----------------------------------------
Monat 8_2_2:
  Anzahl der Aufträge: 127
  Anzahl der Bestellpositionen: 2060
  Anzahl der Maschinen: 65
  Anzahl der Arbeiter: 118
  Anzahl der Anbaugeräte: 368
----------------------------------------
Monat 9_1_2:
  Anzahl der Aufträge: 122
  Anzahl der Bestellpositionen: 1719
  Anzahl der Maschinen: 65
  Anzahl der Arbeiter: 118
  Anzahl der Anbaugeräte: 377
---------------------------------

In [5]:
import json

# Lade die JSON-Datei
with open("Construction_RealLife_2024_7_1_2.json", "r") as f:
    data = json.load(f)

# Extrahiere alle Baustellenstandorte
baustellen_standorte = [
    (item["Standort"]["Item1"], item["Standort"]["Item2"])
    for item in data["Auftraege"]
]

# Extrahiere alle Wohnorte der Arbeiter
arbeiter_wohnorte = [
    (worker["Wohnort"]["Item1"], worker["Wohnort"]["Item2"])
    for worker in data["Arbeiter"]
]

import folium

# Mittelpunkt berechnen (z. B. Mittelwert aller Koordinaten)
all_coords = baustellen_standorte + arbeiter_wohnorte
avg_lat = sum(lat for lat, _ in all_coords) / len(all_coords)
avg_lon = sum(lon for _, lon in all_coords) / len(all_coords)

# Erstelle die Karte zentriert auf den Mittelwert
karte = folium.Map(location=[avg_lat, avg_lon], zoom_start=8)

# Füge Baustellen als blaue Marker hinzu
for lat, lon in baustellen_standorte:
    folium.Marker(
        location=[lat, lon],
        popup="Baustelle",
        icon=folium.Icon(color="blue", icon="wrench", prefix="fa")
    ).add_to(karte)

# Füge Arbeiter als grüne Marker hinzu
for lat, lon in arbeiter_wohnorte:
    folium.Marker(
        location=[lat, lon],
        popup="Arbeiter",
        icon=folium.Icon(color="green", icon="user", prefix="fa")
    ).add_to(karte)

# Karte anzeigen
karte

In [6]:
import json
import folium
import geopandas as gpd
from shapely.geometry import box

# -----------------------------
# 1. Lade deine JSON-Daten
# -----------------------------
with open("Construction_RealLife_2024_7_1_2.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Extrahiere alle Baustellenstandorte
baustellen_standorte = [
    (item["Standort"]["Item1"], item["Standort"]["Item2"])
    for item in data["Auftraege"]
]

# Extrahiere alle Wohnorte der Arbeiter
arbeiter_wohnorte = [
    (worker["Wohnort"]["Item1"], worker["Wohnort"]["Item2"])
    for worker in data["Arbeiter"]
]

# Kombinierte Liste aller Koordinaten (für mittleren Kartenzentroid)
all_coords = baustellen_standorte + arbeiter_wohnorte
avg_lat = sum(lat for lat, _ in all_coords) / len(all_coords)
avg_lon = sum(lon for _, lon in all_coords) / len(all_coords)

# -----------------------------
# 2. Hole Geodaten für Deutschland
# -----------------------------
url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_admin_0_countries.geojson"
world = gpd.read_file(url)

# Deutschland extrahieren
germany = world[world["ADMIN"] == "Germany"].copy()
germany = germany.to_crs("EPSG:4326")

# Weltrechteck als Außenkontur
outer = box(-180, -90, 180, 90)
mask_geometry = outer.difference(germany.geometry.union_all())

# -----------------------------
# 3. Erstelle die Karte
# -----------------------------
karte = folium.Map(location=[avg_lat, avg_lon], zoom_start=6, tiles="CartoDB positron")

'''
# Maske hinzufügen (alles außer Deutschland weiß überdecken)
folium.GeoJson(
    gpd.GeoSeries(mask_geometry).__geo_interface__,
    style_function=lambda x: {
        "fillColor": "white",
        "color": "white",
        "weight": 0,
        "fillOpacity": 1,
    },
    name="Mask"
).add_to(karte)

# Deutschland-Grenzen
folium.GeoJson(
    germany.__geo_interface__,
    style_function=lambda x: {
        "fillColor": "none",
        "color": "black",
        "weight": 1,
    },
    name="Germany"
).add_to(karte)
'''
# -----------------------------
# 4. Füge Marker hinzu
# -----------------------------
# Baustellen (blaue Schraubenschlüssel)
for lat, lon in baustellen_standorte:
    folium.Marker(
        location=[lat, lon],
        popup="Baustelle",
        icon=folium.Icon(color="blue", icon="wrench", prefix="fa")
    ).add_to(karte)

# Arbeiter (graue Personen)
for lat, lon in arbeiter_wohnorte:
    folium.Marker(
        location=[lat, lon],
        popup="Arbeiter",
        icon=folium.Icon(color="gray", icon="user", prefix="fa")
    ).add_to(karte)

# -----------------------------
# 5. Speichern oder anzeigen
# -----------------------------
#karte.save("baustellen_karte_deutschland.html")
karte  # wenn du im Notebook arbeitest